In [1]:
from Pipeline.Algorithm.EvaluationMatrix import EvaluationMatrix
from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Algorithm.EvaluationELM import EvaluationELM
from Pipeline.Global.GlobalSetting import GlobalSetting

In [2]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()

features_size = gallstone_dataset.x_train_scaled.shape[1]

In [3]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x = gallstone_dataset.x_train,
    y = gallstone_dataset.y_train,
    activation_function     = GlobalSetting.sigmoid ,
    elm_initial_state_range = GlobalSetting.elm_initial_state_range ,
    data_split_state_range  = GlobalSetting.data_split_state_range ,
    k_fold=5
)

In [4]:
hidden_size_record ,hidden_size_result = evaluator.grid_search_hidden_size(
    GlobalSetting.hidden_size_explore_range
)
GlobalSetting.save_dataframe_to_record(hidden_size_record,'hidden_size_record.csv')
hidden_node_list = EvaluationELM.extract_top_results(hidden_size_result)
best_hidden_size = hidden_node_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\hidden_size_record.csv


In [5]:
lambda_record, lambda_result = evaluator.grid_search_lambda(
    hidden_size     = best_hidden_size['Hidden_Nodes'],
    lambda_range    = GlobalSetting.lambda_value_explore_range
)
GlobalSetting.save_dataframe_to_record(lambda_record,'lambda_record.csv')
lambda_value_list = EvaluationELM.extract_top_results(lambda_result)
best_lambda_value = lambda_value_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\lambda_record.csv


In [6]:
size_lambda_record , size_lambda_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range   = hidden_node_list['Hidden_Nodes'],
    lambda_range        = GlobalSetting.lambda_value_explore_range
)
GlobalSetting.save_dataframe_to_record(size_lambda_record,'size_lambda_record.csv')
size_lambda_list = EvaluationELM.extract_top_results(size_lambda_result)
best_size_lambda = size_lambda_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\size_lambda_record.csv


In [7]:
model_configs_payload = [
    {
        "Model_Types" : "Best_Hidden_Nodes",
        "Hidden_Nodes": int(best_hidden_size['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_hidden_size['Lambda_Value'])
    },
    {
        "Model_Types" : "Best_Lambda",
        "Hidden_Nodes": int(best_lambda_value['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_lambda_value['Lambda_Value'])
    },
    {
        "Model_Types" : "Best_Size_and_Lambda",
        "Hidden_Nodes": int(best_size_lambda['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_size_lambda['Lambda_Value'])
    }
]
GlobalSetting.upsert_model_configs(model_configs_payload)

In [8]:
lambda_record_elm_seed_eval = EvaluationMatrix.elm_seed_evaluation(lambda_record)
lambda_record_data_seed_eval = EvaluationMatrix.data_seed_evaluation(lambda_record)

In [9]:
lambda_record_elm_seed_eval

,Hidden_Nodes,Activation,Lambda_Value,Accuracy_Pooled_ELM_Seed_Std,Precision_Pooled_ELM_Seed_Std,Recall_Pooled_ELM_Seed_Std,NPV_Pooled_ELM_Seed_Std,Specificity_Pooled_ELM_Seed_Std,F1-Score_Pooled_ELM_Seed_Std,F2-Score_Pooled_ELM_Seed_Std,Bal Accuracy_Pooled_ELM_Seed_Std,MCC_Pooled_ELM_Seed_Std
0,58,sigmoid,0.000977,0.0513,0.0623,0.0685,0.0494,0.0695,0.0573,0.0629,0.0514,0.1034
1,58,sigmoid,0.001953,0.0515,0.0625,0.0683,0.0490,0.0671,0.0584,0.0634,0.0516,0.1038
2,58,sigmoid,0.003906,0.0491,0.0596,0.0627,0.0458,0.0684,0.0541,0.0582,0.0491,0.0985
3,58,sigmoid,0.007812,0.0477,0.0600,0.0609,0.0441,0.0682,0.0523,0.0562,0.0478,0.0960
4,58,sigmoid,0.015625,0.0434,0.0557,0.0502,0.0376,0.0652,0.0461,0.0474,0.0433,0.0868
5,58,sigmoid,0.031250,0.0407,0.0520,0.0497,0.0370,0.0580,0.0435,0.0461,0.0407,0.0818
6,58,sigmoid,0.062500,0.0444,0.0615,0.0559,0.0398,0.0663,0.0484,0.0515,0.0443,0.0897
7,58,sigmoid,0.125000,0.0386,0.0586,0.0563,0.0367,0.0583,0.0447,0.0507,0.0386,0.0787
8,58,sigmoid,0.250000,0.0344,0.0497,0.0546,0.0344,0.0496,0.0418,0.0490,0.0346,0.0702
9,58,sigmoid,0.500000,0.0327,0.0506,0.0572,0.0351,0.0503,0.0402,0.0495,0.0329,0.0676


In [10]:
elm_data_seed_var = lambda_record_elm_seed_eval['F2-Score_Pooled_ELM_Seed_Std'] / lambda_record_data_seed_eval['F2-Score_Pooled_Data_Seed_Std']
elm_data_seed_var

0    0.577594
1    0.583794
2    0.519179
3    0.507220
4    0.421708
5    0.411607
6    0.460644
7    0.449070
8    0.417732
9    0.445946
dtype: float64

In [11]:
size_lambda_list

,Hidden_Nodes,Activation,Lambda_Value,avg_Accuracy_Seed_Mean,avg_Accuracy_Seed_Std,avg_Accuracy_Seed_SEM,avg_Accuracy_Seed_Min,avg_Accuracy_Seed_Max,avg_Precision_Seed_Mean,avg_Precision_Seed_Std,...,avg_Bal Accuracy_Seed_Std,avg_Bal Accuracy_Seed_SEM,avg_Bal Accuracy_Seed_Min,avg_Bal Accuracy_Seed_Max,avg_MCC_Seed_Mean,avg_MCC_Seed_Std,avg_MCC_Seed_SEM,avg_MCC_Seed_Min,avg_MCC_Seed_Max,rank_score
18,57,sigmoid,0.2500,0.7507,0.0054,0.0031,0.5490,0.8824,0.8005,0.0150,...,0.0056,0.0032,0.5477,0.8800,0.5117,0.0124,0.0072,0.0963,0.7858,0.7179
27,56,sigmoid,0.1250,0.7477,0.0106,0.0061,0.5686,0.8824,0.7941,0.0187,...,0.0109,0.0063,0.5662,0.8800,0.5051,0.0229,0.0132,0.1360,0.7858,0.7149
38,55,sigmoid,0.2500,0.7493,0.0070,0.0040,0.5686,0.8824,0.7986,0.0146,...,0.0071,0.0041,0.5662,0.8808,0.5088,0.0158,0.0091,0.1368,0.7858,0.7143
17,57,sigmoid,0.1250,0.7477,0.0044,0.0026,0.5294,0.8824,0.7983,0.0131,...,0.0044,0.0025,0.5269,0.8800,0.5059,0.0113,0.0065,0.0557,0.7858,0.7142
26,56,sigmoid,0.0625,0.7438,0.0079,0.0046,0.5882,0.8824,0.7883,0.0171,...,0.0081,0.0047,0.5862,0.8808,0.4968,0.0192,0.0111,0.1764,0.7735,0.7140
